# Teacher model

Trains the larger model that the deployable student is distilled from in the next stage.

The teacher is never shipped. It exists only to produce soft targets, so its size and speed
do not matter and nothing here needs to fit on a phone. What matters is that it is
meaningfully better than the student, otherwise distillation has nothing to transfer.

**Before running:** phone-verified account, Accelerator GPU T4 x2, Internet on.

EfficientNet-B3 is roughly 3x the parameters of B0 and takes correspondingly longer.
Budget 6 to 8 hours, which is inside the 12 hour session cap but is a large slice of a
weekly quota. Run it once and keep the checkpoint.

## On input resolution

B3 is normally trained at 300px, but this runs at the contract resolution of 224 like
everything else. Distillation requires the teacher and student to see **the same image**,
so a teacher trained at a different resolution would either need a second resize at
distillation time or would score a different crop than the student is learning from. The
teacher gives up a little accuracy for that; the alternative gives up correctness.

In [ ]:
!git clone --depth 1 https://github.com/simonkundrik/plate-vision.git /kaggle/temp/plate-vision
%pip install -q -e "/kaggle/temp/plate-vision/model[train,export]"

import torch

print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit("No GPU. Set Accelerator to GPU T4 x2 before running.")

In [ ]:
%cd /kaggle/temp/plate-vision/model
!python data/download_food101.py --out /kaggle/temp/food101

## Train

Uses whichever recipe won the ablation in notebook 02. Batch size is halved against the
student run because B3 activations are larger and 128 will not fit on a single T4 at 224px.

In [ ]:
!python scripts/train_classifier.py \
    --data-root /kaggle/temp/food101/food-101 \
    --out /kaggle/working/runs/teacher \
    --backbone efficientnet_b3 \
    --epochs 30 --batch-size 64 --lr 7e-4 --weight-decay 0.05 \
    --label-smoothing 0.1 --mixup-alpha 0.2 --cutmix-alpha 1.0 --ema \
    --workers 4 --amp

## Confirm the checkpoint is loadable

Worth doing here rather than discovering it at the start of a distillation run. The
checkpoint records its own backbone, so `restore_classifier` can rebuild the architecture
without being told which one it is. A `state_dict` on its own does not carry that.

In [ ]:
from pathlib import Path

from platevision import checkpoint

teacher, payload = checkpoint.restore_classifier(Path("/kaggle/working/runs/teacher/best.pt"))
print("backbone:   ", payload["backbone"])
print("classes:    ", payload["num_classes"])
print("best top-1: ", round(payload["best_metric"], 2))
print("weights:    ", payload["config"].get("weights", "live"))

teacher.eval()
with torch.no_grad():
    print("output shape:", teacher(torch.zeros(1, 3, 224, 224)).shape)

Download `runs/teacher/best.pt` and upload it as a private Kaggle dataset. The distillation
notebook mounts it rather than retraining the teacher, which is the difference between one
expensive run and one expensive run per distillation experiment.